In [0]:
# Step 1: Define the evaluation set — a list of test questions and their expected answers.
# These cover bonus calculations, employee queries (Genie), multi-tool questions,
# and edge cases like abbreviated numbers ("100k") and word-form numbers.
import pandas as pd

eval_set = pd.DataFrame([
    {
        "question": "What is the bonus for a salary of 100000?",
        "expected": "10000"
    },
    {
        "question": "What is the bonus for a salary of 200000?",
        "expected": "20000"
    },
    {
        "question": "What is the bonus for a salary of 75000?",
        "expected": "7500"
    },
    {
        "question": "How many active employees do we have?",
        "expected": "109"
    },
    {
        "question": "How many employees are active?",
        "expected": "109"
    },
    {
        "question": "How many employees have completed training?",
        "expected": "118"
    },
    {
        "question": "What is the bonus for salary 100k?",
        "expected": "10000"
    },
    {
        "question": "What is the bonus for salary one hundred thousand?",
        "expected": "10000"
    },
    {
        "question": "How many active employees do we have and what is the bonus for a salary of 100000?",
        "expected": "109 and 10000"
    },
    {
        "question": "How many leave balance records are there for the UK region?",
        "expected": "41"
    }
])

eval_set

,question,expected
0,What is the bonus for a salary of 100000?,10000
1,What is the bonus for a salary of 200000?,20000
2,What is the bonus for a salary of 75000?,7500
3,How many active employees do we have?,109
4,How many employees are active?,109
5,How many employees have completed training?,118
6,What is the bonus for salary 100k?,10000
7,What is the bonus for salary one hundred thous...,10000
8,How many active employees do we have and what ...,109 and 10000
9,How many leave balance records are there for t...,41


In [0]:
# Step 2: Import hr_agent (and its dependency genie_tool) from the Multi-Tool HR Agent
# notebook, then run every question in the eval set through the agent.
# The notebook source is exported in Databricks native format (plain text with
# "# COMMAND ----------" delimiters), split by cell, and only function-definition
# cells (containing "def ") are exec'd — test cells are skipped.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat
import base64

w = WorkspaceClient()
_src = base64.b64decode(w.workspace.export(
    "/Users/neethu.mv@tigeranalytics.com/DatabricksAgentCreation/Multi-Tool HR Agent",
    format=ExportFormat.SOURCE
).content).decode("utf-8")

for _c in _src.split("# COMMAND ----------")[1:]:
    if "def " in _c:
        exec(_c.strip())

results = []

for _, row in eval_set.iterrows():

    answer = hr_agent(
        row["question"]
    )

    results.append({
        "question": row["question"],
        "expected": row["expected"],
        "actual": answer
    })

results_df = pd.DataFrame(results)

results_df

,question,expected,actual
0,What is the bonus for a salary of 100000?,10000,Tools Used:\n- calculate_bonus\n\nBonus for sa...
1,What is the bonus for a salary of 200000?,20000,Tools Used:\n- calculate_bonus\n\nBonus for sa...
2,What is the bonus for a salary of 75000?,7500,Tools Used:\n- calculate_bonus\n\nBonus for sa...
3,How many active employees do we have?,109,Tools Used:\n- genie_tool\n\nThere are **109 a...
4,How many employees are active?,109,Tools Used:\n- genie_tool\n\nThere are **109 a...
5,How many employees have completed training?,118,Tools Used:\n- genie_tool\n\nA total of **118 ...
6,What is the bonus for salary 100k?,10000,Tools Used:\n- calculate_bonus\n\nBonus for sa...
7,What is the bonus for salary one hundred thous...,10000,
8,How many active employees do we have and what ...,109 and 10000,Tools Used:\n- calculate_bonus\n- genie_tool\n...
9,How many leave balance records are there for t...,41,Tools Used:\n- genie_tool\n\nThere are **41 le...


In [0]:
# Step 3: Evaluate each result by comparing the agent's actual answer against
# the expected value. The comparison is case-insensitive substring matching.
# The multi-tool question ("109 and 10000") is checked for both values independently.
def evaluate_row(expected, actual):

    actual = str(actual).lower()

    if expected == "109 and 10000":

        return (
            "109" in actual
            and "10000" in actual
        )

    return expected.lower() in actual

results_df["passed"] = results_df.apply(
    lambda r: evaluate_row(
        r["expected"],
        r["actual"]
    ),
    axis=1
)

results_df

,question,expected,actual,passed
0,What is the bonus for a salary of 100000?,10000,Tools Used:\n- calculate_bonus\n\nBonus for sa...,True
1,What is the bonus for a salary of 200000?,20000,Tools Used:\n- calculate_bonus\n\nBonus for sa...,True
2,What is the bonus for a salary of 75000?,7500,Tools Used:\n- calculate_bonus\n\nBonus for sa...,True
3,How many active employees do we have?,109,Tools Used:\n- genie_tool\n\nThere are **109 a...,True
4,How many employees are active?,109,Tools Used:\n- genie_tool\n\nThere are **109 a...,True
5,How many employees have completed training?,118,Tools Used:\n- genie_tool\n\nA total of **118 ...,True
6,What is the bonus for salary 100k?,10000,Tools Used:\n- calculate_bonus\n\nBonus for sa...,False
7,What is the bonus for salary one hundred thous...,10000,,False
8,How many active employees do we have and what ...,109 and 10000,Tools Used:\n- calculate_bonus\n- genie_tool\n...,True
9,How many leave balance records are there for t...,41,Tools Used:\n- genie_tool\n\nThere are **41 le...,True


In [0]:
# Step 4: Calculate the overall evaluation score as the percentage of passed tests.
score = (
    results_df["passed"]
    .mean()
    * 100
)

print(
    f"Evaluation Score: {score:.1f}%"
)

Evaluation Score: 80.0%


In [0]:
# Step 5: Filter and display the questions that failed evaluation.
# These represent gaps in the agent's tool-selection or parameter-extraction logic.
failures = results_df[
    results_df["passed"] == False
]

failures

,question,expected,actual,passed
6,What is the bonus for salary 100k?,10000,Tools Used:\n- calculate_bonus\n\nBonus for sa...,False
7,What is the bonus for salary one hundred thous...,10000,,False


# Step 6: Document the failures and their root causes identified during evaluation.

## Failure Identified During Evaluation

The evaluation process surfaced two failures in bonus calculation requests.

###Failure 1:
Question:
"What is the bonus for salary 100k?"

Expected:
10000

Observed:
The agent incorrectly interpreted the value because it extracted only numeric digits and did not understand the "k" suffix.

Root Cause:
Salary extraction logic does not support abbreviated numeric formats such as 100k or 250k.

---

###Failure 2:
Question:
"What is the bonus for salary one hundred thousand?"

Expected:
10000

Observed:
The agent failed to invoke the bonus-calculation tool.

Root Cause:
The agent relies on regular-expression digit extraction and cannot convert numbers expressed as words into numeric values.

---

Outcome:
Agent Evaluation successfully identified limitations in the tool-selection and parameter-extraction logic prior to deployment.